# Main — Dual-Layer Watermarking Demo App (Gradio, Colab)

Enter a prompt -> get **plain**, **layer-1 (public topic) watermarked** and **dual-layer**
continuations from OPT-2.7b, then run BOTH detectors on every output:

- **Layer 1** (`src/detection/topic_detection.py`): re-infers the topic, scores against its
  static uniform greenlist -> z-score / p-value / confirmed.
- **Layer 2** (`src/detection/kgw_detection.py`): keyed rolling-greenlist ownership check
  -> ownership score / p-value / confirmed.

The UI has two tabs:
- **Generation** — pick `delta_public` / `delta_private` from the dropdown menus, enter a
  prompt, and get the layer-1 and dual-layer generations.
- **Detection** — paste any text and see the detection values for each layer.

Everything imports from `src/` — nothing is redefined here. Runs on a Colab GPU runtime
(T4 is fine; ~15–25 s per prompt since we generate 4 x 200 tokens).

**Public link:** the final cell prints a `https://xxxx.gradio.live` URL — anyone can
open the app from it while this Colab session is alive. The link dies when the runtime
disconnects, so re-run the last cell to get a fresh one for each demo.


In [1]:
print("Hello world")

Hello world


In [9]:
!ls

choices.md  model      plan_eval.md  README.md	       setup.md
data	    notebooks  PLAN.md	     requirements.txt  src


In [8]:
%cd Dual_watermarking_Scheme

/content/Dual_watermarking_Scheme


In [2]:
!git clone https://github.com/pravaspaudel/Dual_watermarking_Scheme.git
%cd Dual_watermarking_Scheme
!pip install -q transformers scipy pandas accelerate gradio

Cloning into 'Dual_watermarking_Scheme'...
remote: Enumerating objects: 317, done.
remote: Counting objects: 100% (166/166), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 317 (delta 78), reused 115 (delta 33), pack-reused 151 (from 1)
Receiving objects: 100% (317/317), 81.98 MiB | 31.78 MiB/s, done.
Resolving deltas: 100% (137/137), done.
/content/Dual_watermarking_Scheme


In [10]:
from huggingface_hub import login
login()

In [11]:
import os
import torch
import pandas as pd

from src.utils.model import load_model
from src.utils.loadConfig import load_config
from src.watermark.dual_layer import DualWaterMarking
from src.detection.topic_detection import prepare, detect_topic_watermark
from src.detection.kgw_detection import detect_private_watermark
from src.utils.key_manager import generate_key

config = load_config("secondLayer")
key = generate_key()
print(config)

{'MODEL_NAME': 'facebook/opt-2.7b', 'GREEN_FRACTION': 0.5, 'PREV_TOKEN_SIZE': 5, 'DETECTION_THRESHOLD': 0.6, 'P_VALUE_THRESHOLD': 0.05}


In [12]:
# src/utils/model.py loads to CPU by default -- other notebooks always move it
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

model, tokenizer, VOCAB_SIZE = load_model(config["MODEL_NAME"])
model.to(DEVICE).eval()

DELTA_PUBLIC_CHOICES = [1.0, 1.5, 2.0, 2.5, 3.0]
DELTA_PRIVATE_CHOICES = [0.3, 0.5, 0.7, 1.0, 1.5]

wm = None


def build_wm(delta_public=2.0, delta_private=0.7):
    """(re)create the watermarking wrapper with the chosen deltas"""
    global wm
    if wm is not None:
        del wm
        torch.cuda.empty_cache()
    wm = DualWaterMarking(
        model,
        tokenizer,
        greenlist_dir="data/greenlist",
        split="all",
        green_fraction=config["GREEN_FRACTION"],
        prev_token_size=config["PREV_TOKEN_SIZE"],
        delta_public=float(delta_public),
        delta_private=float(delta_private),
        max_new_tokens=200,
        seed=0,
    )
    return wm


wm = build_wm()
state = prepare(model, tokenizer, greenlist_dir="data/greenlist", split="all")

print("topics:", wm.topics)
print("key fingerprint:", wm.key[:8], "...")

device: cuda


Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 5.30GB            

model.safetensors: downloading bytes:           |  0.00B            

model and tokenizer of facebook/opt-2.7b loaded with vocab_size 50265
topics: ['entertainment', 'finance', 'history', 'medicine', 'politics', 'science', 'sports', 'technology']
key fingerprint: 0d01fad9 ...


## Generate + full detection for one prompt

In [16]:
def analyze(prompt):
    """one prompt -> dict with all three generations and every detector verdict"""
    row = wm.watermark([prompt], include_single_layers=True).iloc[0]
    texts = {
        "plain":   row["plain_output"],
        "layer1_only_output": row["layer1_only_output"],
        "layer2_only_output": row["layer2_only_output"],
        "dual_watermarked_output": row["dual_watermarked_output"],
    }
    det1 = {name: detect_topic_watermark(text, tokenizer, state, vocab_size=VOCAB_SIZE)
            for name, text in texts.items()}
    det2 = {
        name: detect_private_watermark(
            text=text,
            tokenizer=tokenizer,
            key=wm.key,
            vocab_size=VOCAB_SIZE,
            green_fraction=config["GREEN_FRACTION"],
            prev_token_size=config["PREV_TOKEN_SIZE"],
            threshold=config["DETECTION_THRESHOLD"],
            p_value_threshold=config["P_VALUE_THRESHOLD"],
        )
        for name, text in texts.items()
    }
    return texts, det1, det2, row["topic"]

In [19]:
import gradio as gr

OK = "CONFIRMED"
BAD = "REJECTED"
ARROW = "->"


def _fmt_verdict(det1_res, det2_res):
    v1 = OK if det1_res["confirmed"] else "NOT CONFIRMED"
    v2 = ("OWNED" + " [" + OK + "]") if det2_res["confirmed"] else ("NOT OWNED [REJECTED]")
    return (
        f"| statistic | value |\n"
        f"|---|---|\n"
        f"| L1 detected topic | **{det1_res['topic']}** (cos {det1_res['topic_score']}) |\n"
        f"| L1 z-score | **{det1_res['z_score']}** {ARROW} {v1} |\n"
        f"| L1 green hits | {det1_res['match_count']} / {det1_res['num_positions']} "
        f"(gamma {det1_res['gamma']}) |\n"
        f"| L2 ownership score | **{round(det2_res['ownership_score'], 3)}** {ARROW} {v2} |\n"
        f"| L2 green hits | {det2_res['match_count']} / {det2_res['num_positions']} |"
    )


def _detect_text(text, key_used):
    """run both detectors on arbitrary text using the given key -> (l1 md, l2 md, both md)"""
    d1 = detect_topic_watermark(text, tokenizer, state, vocab_size=VOCAB_SIZE)
    d2 = detect_private_watermark(
        text=text,
        tokenizer=tokenizer,
        key=key_used,
        vocab_size=VOCAB_SIZE,
        green_fraction=config["GREEN_FRACTION"],
        prev_token_size=config["PREV_TOKEN_SIZE"],
        threshold=config["DETECTION_THRESHOLD"],
        p_value_threshold=config["P_VALUE_THRESHOLD"],
    )
    v1 = OK if d1["confirmed"] else "NOT CONFIRMED"
    l1_md = (
        "| Layer 1 (public topic) statistic | value |\n|---|---|\n"
        f"| detected topic | **{d1['topic']}** (cos {d1['topic_score']}) |\n"
        f"| z-score | **{d1['z_score']}** {ARROW} **{v1}** |\n"
        f"| p-value | {d1.get('p_value', '-')} |\n"
        f"| green hits | {d1['match_count']} / {d1['num_positions']} (gamma {d1['gamma']}) |"
    )
    v2 = OK if d2["confirmed"] else "NOT CONFIRMED"
    key_match = "MATCHES session key" if key_used == wm.key else "DOES NOT MATCH session key"
    l2_md = (
        "| Layer 2 (private KGW) statistic | value |\n|---|---|\n"
        f"| ownership score | **{round(d2['ownership_score'], 3)}** {ARROW} **{v2}** |\n"
        f"| p-value | {d2.get('p_value', '-')} |\n"
        f"| green hits | {d2['match_count']} / {d2['num_positions']} |\n"
        f"| threshold | {config['DETECTION_THRESHOLD']} (p <= {config['P_VALUE_THRESHOLD']}) |\n"
        f"| key used | {key_match} |"
    )

    both_md = (
    "**Dual (both layers)**\n\n"
    + _fmt_verdict(d1, d2)
    + f"\n\n### Both-layers claim: {'CONFIRMED' if (d1['confirmed'] and d2['confirmed']) else 'REJECTED'}"
    + "\n\n(topic AND private ownership must both pass)"
    )
    return l1_md, l2_md, both_md


def run_generation(prompt, delta_public, delta_private):
    build_wm(delta_public=delta_public, delta_private=delta_private)
    texts, det1, det2, topic = analyze(prompt)

    plain_md = f"**Routed topic:** `{topic}`"

    l1_md = "**Layer-1 only (public topic watermark)**"

    l2_md = "**Layer-2 only (private keyed watermark)**"

    dual_md = "**Dual (both layers)**"


    return texts["plain"], texts["layer1_only_output"], texts["layer2_only_output"], \
           texts["dual_watermarked_output"], plain_md, l1_md, l2_md, dual_md


def run_detection(text, key_input):
    key_used = key_input if key_input else wm.key
    return _detect_text(text, key_used)


with gr.Blocks(title="Dual-Layer Watermarking Demo") as demo:
    gr.Markdown("## Dual-Layer Watermarking — OPT-2.7b\n"
                "*Layer 1:* public topic greenlist boost · "
                "*Layer 2:* private keyed KGW boost")

    with gr.Tab("Generation"):
        with gr.Row():
            delta_pub_in = gr.Dropdown(
                choices=[str(v) for v in DELTA_PUBLIC_CHOICES],
                value=str(2.0), label="delta_public", info="public topic-layer boost strength")
            delta_priv_in = gr.Dropdown(
                choices=[str(v) for v in DELTA_PRIVATE_CHOICES],
                value=str(0.7), label="delta_private", info="private keyed-layer boost strength")
        prompt_in = gr.Textbox(label="Prompt", lines=3,
                               placeholder="e.g. The government announced a new policy on...")
        btn_gen = gr.Button("Generate & Detect", variant="primary")

        out_plain = gr.Textbox(label="Normal response", lines=6, interactive=False)
        out_l1 = gr.Textbox(label="Layer-1 watermarked response", lines=6, interactive=False)
        out_l2 = gr.Textbox(label="Layer-2 watermarked response", lines=6, interactive=False)
        out_dual = gr.Textbox(label="Dual-layer watermarked response", lines=6, interactive=False)

        rep_plain = gr.Markdown()
        rep_l1 = gr.Markdown()
        rep_l2 = gr.Markdown()
        rep_dual = gr.Markdown()

        btn_gen.click(run_generation,
                      inputs=[prompt_in, delta_pub_in, delta_priv_in],
                      outputs=[out_plain, out_l1, out_l2, out_dual,
                               rep_plain, rep_l1, rep_l2, rep_dual])

        gr.Examples(
            examples=[
                ["The new vaccine trial results were published today and"],
                ["The central bank raised interest rates because"],
                ["The football team won the championship after"],
            ],
            inputs=prompt_in,
        )

    with gr.Tab("Detection"):
        gr.Markdown(f"**Session watermark key (testing only):** `{wm.key}`")

        det_in = gr.Textbox(label="Text to check", lines=8,
                            placeholder="Paste generated (or any other) text here...")
        key_in = gr.Textbox(label="Secret key (testing only)", type="password",
                            info="Leave empty to use the session key above, "
                                 "or supply one to test detection with a different key.")
        btn_det = gr.Button("Run Detection", variant="primary")

        det_l1 = gr.Markdown(label="Layer 1 results")
        det_l2 = gr.Markdown(label="Layer 2 results")
        det_both = gr.Markdown()

        btn_det.click(run_detection, inputs=[det_in, key_in],
                      outputs=[det_l1, det_l2, det_both])

        gr.Examples(
            examples=[
                ["The new vaccine trial results were published today and showed strong efficacy across all cohorts.", ""],
                ["The government announced a new policy on renewable energy subsidies earlier this week.", ""],
            ],
            inputs=[det_in, key_in],
        )

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ca2c9eda4496b623c8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
